# 10.5 怎麼學到可用的視覺特徵？


一個入口能把紅方塊變成16個向量，還不表示其中保留了可以讀出的顏色與形狀。我們可以先給它一件小任務：兩張圖片一張紅方塊、一張藍圓形，讓它預測各自編號。預測錯的代價會沿計算返回，告訴像素變換哪些數字應該調整。這是視覺特徵開始獲得任務意義的來源。

前置是[10.3的像素變換](https://birdhackor.github.io/tiny-perceptron-vlm/10.3.html)、[1.8分類猜錯代價](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html)與[W.6梯度](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)。encoder是編碼器，負責把原始圖變成特徵；classifier是分類頭，把特徵轉成每種候選的分數。交叉熵CE比較分數與正確編號，梯度是這些數字對代價的敏感程度。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from tiny_perceptron.multimodal import VisionEncoder, scene

torch.manual_seed(0)
encoder = VisionEncoder(width=8)
classifier = nn.Linear(8, 2)
images = torch.stack([scene("red", "square"), scene("blue", "circle")])
features = encoder(images)
logits = classifier(features.mean(1))
loss = F.cross_entropy(logits, torch.tensor([0, 1]))
loss.backward()
print("特徵", tuple(features.shape), "分數", tuple(logits.shape))
print("入口收到梯度", encoder.projection.weight.grad.norm().item() > 0)

輸入兩張合成圖，由`stack`堆成一批。`VisionEncoder(width=8)`依序切patch、做線性變換、加位置並處理特徵，輸出`(2,16,8)`。`mean(1)`把每張圖16個位置平均成8個值，再讓分類頭輸出`(2,2)`，即兩張圖各有兩個類別分數。正確編號[0,1]來自我們生成圖片時的設定，不是模型猜出的標籤。

`backward()`把代價的敏感程度傳回所有參與計算的可訓練參數；最後一行檢查入口矩陣梯度長度大於0，正常輸出True。這裡沒有執行參數更新，不能稱為已訓練成功。即使反復訓練這兩張圖，它還可能只是記住兩張樣本，或用顏色代替形狀，因為紅總是方塊、藍總是圓。

真正驗證要有更多顏色與形狀組合，並保留未訓練的組合或生成位置作為測試。之後可拿掉分類頭、保留編碼器特徵，連接語言模型；任務改變也可能需要繼續調整。

我們已把這一步做成[正式編碼器實驗](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/encoders.json)。它和上面的兩圖梯度示範不同：紅、綠、藍各配圓與方，共六個候選，答案例如`red circle`。每張圖仍是16×16 RGB、像素範圍0到1、patch邊長4；編碼器寬度改16，平均16條特徵後，由分類頭在六個候選中選一個。編碼器與分類頭合計1,446個參數，一起更新250次，每次抽八張訓練圖，種子42、學習率0.003。

這次不是保留新的顏色形狀組合，而是保留新的位置：位移−2、−1、0的18張圖用來訓練，位移1的六張圖用來驗證，位移2的六張圖留到最後測試。相同顏色形狀在三邊都出現，位置卻不同，所以它回答的是「已知六類能否在這些新位置辨別」，沒有測過未知物件。更新前只答對1/6，更新後驗證5/6、測試6/6；同一固定訓練小批的交叉熵由1.8468降至0.0173，權重也實際改變。這比只印出梯度多了真正更新與留出題的證據。

六張測試全對值得保留，但六張仍很少，而且位移1反而有一張錯誤。不能把位置更遠就必定更難、某一批全對就一定可靠，當成模型的普遍規律。這像練過六種積木後，在另一格桌面認對同樣六種積木，還沒有認過衣服、動物或手寫字。接頭接下來拿走的是已學特徵；分類頭的6/6也沒有保證語言模型能把它們寫成正確句子。[11.3](https://birdhackor.github.io/tiny-perceptron-vlm/11.3.html)會再測那一段連接。

[真實資料小實驗](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/real_modal.json)另用了Fashion-MNIST的50張服飾圖，不沿用上面合成圖的已訓練編碼器。十類各五張，按每類前三張教學、第四張驗證、第五張測試，得到30／10／10；原始50張都選自上游training部分，這是我們自己的切分，並非官方測試成績。28×28灰階先轉成三個相同灰階通道的RGB，再縮成16×16，像素除255放到0至1；沒有再按每張圖的平均與標準差重新調整像素。

新視覺入口接直接SFT文字底座。SFT是監督式微調：用人寫的提問與正確回答，教模型在回答位置預測目標文字；「直接」表示從新起點直接讀小世界的屬性問答，例如先列出顏色與形狀，再問形狀，目標回答`circle`，沒有另做一般文字續寫階段。「文字底座」就是已受過這段問答訓練的文字模型，兩條路線的起點可回看[7.11](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html)。這次把視覺入口和文字模型全部開放更新250次，累計7,528個有效回答目標，回答`clothing?`的英文類別名稱。固定訓練探針代價從8.968降到0.154，驗證4/10、測試3/10；十道測試都以EOS結束，卻只有Ankle boot、Sneaker、Sandal三張完整答對。比如T-shirt/top、Dress、Pullover都被答成Bag，真實Bag又被答成Shirt。這批只完成很小的服飾命名檢查，沒有自然照片的描述或問答證據；從圓方積木轉到衣物輪廓，不能把原先6/6直接帶過來。

這兩張圖的編號0、1只是兩個候選的名字，尚未讓任務單獨辨別「圓形」。想教形狀，資料要有紅圓、藍圓都標圓、紅方、藍方都標方；想教顏色，則相同顏色不同形狀共用顏色標籤。把任務寫清後，才能判斷平均16個位置的特徵是否夠用。例如只問顏色，可以對RGB各通道分別取空間平均，保留三個色彩數值，或平均已學到的各條色彩特徵；把所有通道再合成一個亮度，可能讓同樣大小的紅、藍圖得到相同數字。問左右位置時，空間平均也可能丟失排列。這一節的分類頭是建立可核對監督的工具，不保證同一個平均表示足以完成之後每一種視覺問題。

練習只把正確編號[0,1]換成[1,0]，先說明新目標在教相反的映射，再執行確認仍能收到梯度。梯度能通過不代表標籤正確，這是準備資料時不能省略人工核對的原因。
